# ⚡ SOLUCIÓN — Taller Performance Testing
## ⚠️ Uso exclusivo del docente

**Docente:** Mg. Sergio Alejandro Burbano Mena  
**Universidad:** Universitaria de Colombia  
**Materia:** Pruebas de Software | 7mo Semestre

---

In [ ]:
!pip install requests locust matplotlib pandas --quiet

import time, threading, statistics, random, json
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from datetime import date

print('✅ Setup completo.')

In [ ]:
# ══ Simulador de API (idéntico al taller) ══════════════════════

class SimuladorAPISistemaNotas:
    def __init__(self):
        self._peticiones_activas = 0
        self._lock = threading.Lock()
        self._total_peticiones = 0
        self._errores = 0

    def _simular_respuesta(self, tiempo_base_ms, factor_carga=True):
        with self._lock:
            self._peticiones_activas += 1
            carga = self._peticiones_activas
            self._total_peticiones += 1
        try:
            if factor_carga and carga > 10:
                factor = 1 + (carga - 10) * 0.15
                tiempo_base_ms = tiempo_base_ms * factor
            variacion = random.uniform(0.7, 1.3)
            tiempo_real = tiempo_base_ms * variacion
            if random.random() < 0.005:
                with self._lock:
                    self._errores += 1
                time.sleep(tiempo_real / 1000)
                return {'status': 500, 'error': 'Internal Server Error', 'tiempo_ms': tiempo_real}
            time.sleep(tiempo_real / 1000)
            return {'status': 200, 'tiempo_ms': tiempo_real}
        finally:
            with self._lock:
                self._peticiones_activas -= 1

    def get_health(self):           return self._simular_respuesta(15, False)
    def listar_estudiantes(self):   return self._simular_respuesta(85)
    def consultar_notas_estudiante(self, _id): return self._simular_respuesta(120)
    def buscar_notas(self, query):  return self._simular_respuesta(800)
    def registrar_nota(self, *a):   return self._simular_respuesta(200)
    def generar_reporte(self, sem): return self._simular_respuesta(2500)
    def get_stats(self):
        return {'total': self._total_peticiones, 'errores': self._errores,
                'error_rate_pct': round(self._errores/max(1,self._total_peticiones)*100,2)}
    def reset_stats(self):
        with self._lock:
            self._total_peticiones = 0; self._errores = 0

api = SimuladorAPISistemaNotas()
print('✅ Simulador creado.')

In [ ]:
# ══ Funciones de medición ════════════════════════════════════

def medir_endpoint(funcion_endpoint, n_veces=50):
    tiempos = []; errores = 0
    for _ in range(n_veces):
        inicio = time.perf_counter()
        respuesta = funcion_endpoint()
        fin = time.perf_counter()
        tiempos.append((fin - inicio) * 1000)
        if respuesta['status'] != 200: errores += 1
    tiempos.sort()
    n = len(tiempos)
    return {'n_peticiones': n_veces, 'errores': errores,
            'error_rate': round(errores/n_veces*100,2),
            'min_ms': round(min(tiempos),1), 'max_ms': round(max(tiempos),1),
            'avg_ms': round(statistics.mean(tiempos),1),
            'p50_ms': round(tiempos[int(n*0.50)],1),
            'p90_ms': round(tiempos[int(n*0.90)],1),
            'p95_ms': round(tiempos[int(n*0.95)],1),
            'p99_ms': round(tiempos[min(int(n*0.99),n-1)],1),
            'tiempos_raw': tiempos}

def imprimir_resultados(nombre, r, sla=500.0):
    cumple = r['p95_ms'] <= sla
    estado = '✅ CUMPLE SLA' if cumple else '❌ VIOLA SLA'
    print(f'\n{"═"*55}\n  {nombre}\n{"═"*55}')
    print(f'  Reqs: {r["n_peticiones"]} | Errores: {r["errores"]} ({r["error_rate"]}%)')
    print(f'  p50: {r["p50_ms"]}ms | p90: {r["p90_ms"]}ms | p95: {r["p95_ms"]}ms → {estado}')
    print(f'  p99: {r["p99_ms"]}ms | Max: {r["max_ms"]}ms\n{"═"*55}')

print('✅ Funciones listas.')

In [ ]:
# ══ Baseline sin carga ════════════════════════════════════════

print('📊 Baseline (secuencial)...')
api.reset_stats()

endpoints_baseline = {
    'GET /api/health':          lambda: api.get_health(),
    'GET /api/estudiantes':     lambda: api.listar_estudiantes(),
    'GET /api/notas/{id}':      lambda: api.consultar_notas_estudiante('2024-001'),
    'GET /api/notas/buscar':    lambda: api.buscar_notas('pruebas de software'),
    'POST /api/notas':          lambda: api.registrar_nota('2024-001','INF-701',4.5),
}

resultados_baseline = {}
for nombre, fn in endpoints_baseline.items():
    print(f'   Midiendo {nombre}...')
    resultados_baseline[nombre] = medir_endpoint(fn, n_veces=30)
    imprimir_resultados(nombre, resultados_baseline[nombre])

print('\n✅ Baseline completo.')

In [ ]:
# ══ Load Test — implementación completa ══════════════════════

def load_test(funcion_endpoint, n_usuarios, duracion_s, ramp_up_s=5.0):
    tiempos_todos = []; errores_todos = []
    _lock = threading.Lock()
    fin_del_test = time.time() + duracion_s

    def simular_usuario(usuario_id):
        while time.time() < fin_del_test:
            inicio = time.perf_counter()
            try:
                # ✅ SOLUCIÓN: llamar al endpoint y registrar el resultado
                respuesta = funcion_endpoint()
                tiempo_ms = (time.perf_counter() - inicio) * 1000
                with _lock:
                    tiempos_todos.append(tiempo_ms)
                    if respuesta.get('status', 0) != 200:
                        errores_todos.append({'tiempo_ms': tiempo_ms, 'status': respuesta.get('status')})
            except Exception as e:
                tiempo_ms = (time.perf_counter() - inicio) * 1000
                with _lock:
                    tiempos_todos.append(tiempo_ms)
                    errores_todos.append({'error': str(e)})
            time.sleep(random.uniform(0.1, 0.5))

    hilos = []; inicio_test = time.time()
    for i in range(n_usuarios):
        time.sleep(ramp_up_s / max(n_usuarios, 1) / n_usuarios)
        hilo = threading.Thread(target=simular_usuario, args=(i,), daemon=True)
        hilos.append(hilo); hilo.start()
    for hilo in hilos:
        hilo.join(timeout=duracion_s + 10)

    duracion_real = time.time() - inicio_test
    if not tiempos_todos:
        return {'error': 'No se recolectaron datos'}
    tiempos_todos.sort()
    n_total = len(tiempos_todos)
    n_errores = len(errores_todos)
    return {
        'n_usuarios_vus': n_usuarios, 'duracion_s': round(duracion_real,1),
        'n_peticiones': n_total, 'n_errores': n_errores,
        'error_rate': round(n_errores/max(1,n_total)*100,2),
        'throughput_rps': round(n_total/duracion_real,1),
        'p50_ms': round(tiempos_todos[int(n_total*0.50)],1),
        'p90_ms': round(tiempos_todos[int(n_total*0.90)],1),
        'p95_ms': round(tiempos_todos[int(n_total*0.95)],1),
        'p99_ms': round(tiempos_todos[min(int(n_total*0.99),n_total-1)],1),
        'max_ms': round(max(tiempos_todos),1),
        'tiempos_raw': tiempos_todos
    }

print('✅ load_test() implementado.')

In [ ]:
# ══ Ejecutar Load Test ════════════════════════════════════════

print('🏋️ Ejecutando Load Test: 50 VUs | 60s...')
api.reset_stats()

resultado_load = load_test(
    funcion_endpoint=lambda: api.consultar_notas_estudiante('2024-001'),
    n_usuarios=50,
    duracion_s=60,
    ramp_up_s=10.0
)

print('\n📊 RESULTADOS LOAD TEST')
print(f'   VUs: {resultado_load["n_usuarios_vus"]} | Duración: {resultado_load["duracion_s"]}s')
print(f'   Peticiones: {resultado_load["n_peticiones"]} | Throughput: {resultado_load["throughput_rps"]} req/s')
print(f'   Error rate: {resultado_load["error_rate"]}% {"✅" if resultado_load["error_rate"] < 1 else "❌"}')
print(f'   p50: {resultado_load["p50_ms"]}ms | p95: {resultado_load["p95_ms"]}ms {"✅" if resultado_load["p95_ms"] < 500 else "❌"} | p99: {resultado_load["p99_ms"]}ms')

In [ ]:
# ══ Gráfica Baseline vs Load Test ════════════════════════════

endpoint_key = 'GET /api/notas/{id}'
baseline = resultados_baseline[endpoint_key]
load = resultado_load

percentiles = ['p50', 'p90', 'p95', 'p99']
vals_baseline = [baseline['p50_ms'], baseline['p90_ms'], baseline['p95_ms'], baseline['p99_ms']]
vals_load     = [load['p50_ms'],     load['p90_ms'],     load['p95_ms'],     load['p99_ms']]

x = range(len(percentiles))
ancho = 0.35

fig, ax = plt.subplots(figsize=(9, 5))
bars1 = ax.bar([i - ancho/2 for i in x], vals_baseline, ancho, label='Baseline (1 usuario)', color='#059669', alpha=0.85)
bars2 = ax.bar([i + ancho/2 for i in x], vals_load,     ancho, label='Load Test (50 VUs)',   color='#EA580C', alpha=0.85)

# Línea del SLA
ax.axhline(y=500, color='red', linestyle='--', linewidth=2, label='SLA p95 = 500ms')

# Etiquetas de valores sobre las barras
for bar in bars1: ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+5, f'{bar.get_height():.0f}ms', ha='center', va='bottom', fontsize=9, color='#065F46')
for bar in bars2: ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+5, f'{bar.get_height():.0f}ms', ha='center', va='bottom', fontsize=9, color='#9A3412')

ax.set_xlabel('Percentil', fontsize=12)
ax.set_ylabel('Tiempo de respuesta (ms)', fontsize=12)
ax.set_title(f'Baseline vs Load Test — {endpoint_key}', fontsize=13, fontweight='bold')
ax.set_xticks(list(x)); ax.set_xticklabels(percentiles)
ax.legend(); ax.grid(axis='y', alpha=0.3)
plt.tight_layout(); plt.show()

print(f'\nDegradación con 50 VUs:')
for p, b, l in zip(percentiles, vals_baseline, vals_load):
    delta = ((l-b)/b)*100
    print(f'  {p}: {b}ms → {l}ms  (+{delta:.0f}%)')

In [ ]:
# ══ Stress Test progresivo ═══════════════════════════════════

def stress_test_progresivo(funcion_endpoint, niveles_vus, duracion_por_nivel_s=30):
    resultados = []
    for n_vus in niveles_vus:
        print(f'   Probando {n_vus} VUs...', end=' ')
        r = load_test(funcion_endpoint, n_usuarios=n_vus, duracion_s=duracion_por_nivel_s)
        if r and 'error' not in r:
            r['n_vus'] = n_vus
            resultados.append(r)
            estado = '✅' if r['p95_ms'] < 500 else '❌'
            print(f'p95={r["p95_ms"]}ms {estado} | err={r["error_rate"]}% | {r["throughput_rps"]}req/s')
        else:
            print('Sin datos')
    return resultados


print('🔥 Stress Test: 10→50→100→200→300 VUs')
niveles = [10, 50, 100, 200, 300]
resultados_stress = stress_test_progresivo(
    lambda: api.consultar_notas_estudiante('2024-001'),
    niveles_vus=niveles,
    duracion_por_nivel_s=30
)
print('\n✅ Stress test completo.')

In [ ]:
# ══ Curva de capacidad ═══════════════════════════════════════

if resultados_stress:
    vus    = [r['n_vus']           for r in resultados_stress]
    p95s   = [r['p95_ms']          for r in resultados_stress]
    tps    = [r['throughput_rps']  for r in resultados_stress]
    errors = [r['error_rate']      for r in resultados_stress]

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

    # ── Gráfica 1: VUs vs p95 ─────────────────────────────────
    colores = ['#16A34A' if p < 500 else '#DC2626' for p in p95s]
    ax1.fill_between(vus, 0, 500, alpha=0.08, color='green', label='Zona SLA OK')
    ax1.fill_between(vus, 500, max(max(p95s)*1.1, 600), alpha=0.08, color='red', label='Zona SLA Violado')
    ax1.plot(vus, p95s, 'o-', color='#1E293B', linewidth=2, markersize=8, zorder=5)
    for v, p, c in zip(vus, p95s, colores):
        ax1.plot(v, p, 'o', color=c, markersize=10, zorder=6)
        ax1.annotate(f'{p}ms', (v, p), textcoords='offset points', xytext=(0,10), ha='center', fontsize=9)
    ax1.axhline(500, color='red', linestyle='--', linewidth=2, label='SLA: 500ms')
    ax1.set_xlabel('Usuarios Concurrentes (VUs)', fontsize=11)
    ax1.set_ylabel('p95 Response Time (ms)', fontsize=11)
    ax1.set_title('Curva de Degradación — p95 vs VUs', fontsize=12, fontweight='bold')
    ax1.legend(); ax1.grid(alpha=0.3)

    # ── Gráfica 2: VUs vs Throughput ──────────────────────────
    ax2.plot(vus, tps, 's-', color='#2563EB', linewidth=2, markersize=8)
    for v, t in zip(vus, tps):
        ax2.annotate(f'{t}', (v, t), textcoords='offset points', xytext=(0,8), ha='center', fontsize=9, color='#1E40AF')
    ax2.set_xlabel('Usuarios Concurrentes (VUs)', fontsize=11)
    ax2.set_ylabel('Throughput (req/s)', fontsize=11)
    ax2.set_title('Throughput vs VUs', fontsize=12, fontweight='bold')
    ax2.grid(alpha=0.3)

    plt.suptitle('Stress Test — Sistema de Notas', fontsize=14, fontweight='bold', y=1.02)
    plt.tight_layout(); plt.show()

    # Encontrar breaking point
    breaking_point = next((r['n_vus'] for r in resultados_stress if r['p95_ms'] > 500), None)
    print(f'\n🔴 Breaking point: {breaking_point} VUs (p95 supera 500ms)' if breaking_point else '\n✅ No se encontró breaking point en los niveles probados')

In [ ]:
# ══ Comparación de todos los endpoints ═══════════════════════

print('📊 Midiendo todos los endpoints con 50 VUs...')

endpoints_completos = {
    'GET /api/health':          lambda: api.get_health(),
    'GET /api/estudiantes':     lambda: api.listar_estudiantes(),
    'GET /api/notas/{id}':      lambda: api.consultar_notas_estudiante('2024-001'),
    'GET /api/notas/buscar':    lambda: api.buscar_notas('pruebas de software'),
    'POST /api/notas':          lambda: api.registrar_nota('2024-001','INF-701',4.5),
    'GET /api/reportes/{sem}':  lambda: api.generar_reporte('2024-2'),
}

resultados_carga_completa = {}
for nombre, fn in endpoints_completos.items():
    print(f'   Midiendo {nombre}...')
    resultados_carga_completa[nombre] = load_test(fn, n_usuarios=50, duracion_s=30)
    r = resultados_carga_completa[nombre]
    if 'error' not in r:
        estado = '✅' if r['p95_ms'] < 500 else '❌'
        print(f'      p95={r["p95_ms"]}ms {estado} | err={r["error_rate"]}% | {r["throughput_rps"]}req/s')

print('\n✅ Todos los endpoints medidos.')

In [ ]:
# ══ Gráfica comparativa de endpoints ════════════════════════

nombres = list(resultados_carga_completa.keys())
p95s_endpoints = []
colores_endpoints = []

for nombre in nombres:
    r = resultados_carga_completa[nombre]
    p95 = r.get('p95_ms', 0) if 'error' not in r else 0
    p95s_endpoints.append(p95)
    colores_endpoints.append('#16A34A' if p95 < 500 else '#DC2626')

fig, ax = plt.subplots(figsize=(10, 6))
bars = ax.barh(nombres, p95s_endpoints, color=colores_endpoints, alpha=0.85, edgecolor='white')
ax.axvline(500, color='red', linestyle='--', linewidth=2, label='SLA: 500ms')

for bar, val in zip(bars, p95s_endpoints):
    ax.text(val + 20, bar.get_y() + bar.get_height()/2, f'{val:.0f}ms',
            va='center', fontsize=10, fontweight='bold')

green_patch = mpatches.Patch(color='#16A34A', label='Cumple SLA (p95 < 500ms)')
red_patch   = mpatches.Patch(color='#DC2626', label='Viola SLA (p95 > 500ms)')
ax.legend(handles=[green_patch, red_patch, plt.Line2D([0],[0],color='red',linestyle='--',label='SLA 500ms')])

ax.set_xlabel('p95 Response Time (ms)', fontsize=12)
ax.set_title('Comparativa p95 por Endpoint (50 VUs)', fontsize=13, fontweight='bold')
ax.grid(axis='x', alpha=0.3)
plt.tight_layout(); plt.show()

In [ ]:
# ══ Resumen Ejecutivo ════════════════════════════════════════

# Recolectar datos para el reporte
if resultado_load and resultados_stress and resultados_carga_completa:
    breaking_point = next((r['n_vus'] for r in resultados_stress if r['p95_ms'] > 500), 'No encontrado')
    endpoints_fallidos = [(n, r['p95_ms']) for n, r in resultados_carga_completa.items()
                          if 'error' not in r and r['p95_ms'] > 500]
    throughput = resultado_load['throughput_rps']
    p95_global = resultado_load['p95_ms']
    error_rate = resultado_load['error_rate']
    listo = len(endpoints_fallidos) == 0 and error_rate < 1

    print(f"""
╔══════════════════════════════════════════════════════════════╗
║  RESUMEN EJECUTIVO — PERFORMANCE TEST REPORT                ║
║  Sistema: API Sistema de Notas Universitario                 ║
╠══════════════════════════════════════════════════════════════╣
║  Fecha: {date.today()}  |  Docente: Mg. Burbano Mena          ║
╠══════════════════════════════════════════════════════════════╣
║  MÉTRICAS BAJO CARGA (50 VUs / 60 segundos)                  ║
║  Throughput:  {throughput:<10} req/s                            ║
║  Error rate:  {error_rate:<10} %  {'✅' if error_rate < 1 else '❌'}                                ║
║  p95 global:  {p95_global:<10} ms {'✅' if p95_global < 500 else '❌'}                               ║
╠══════════════════════════════════════════════════════════════╣
║  BREAKING POINT: {str(breaking_point)+' VUs':<42}║
╠══════════════════════════════════════════════════════════════╣
║  ENDPOINTS PROBLEMÁTICOS                                     ║""")
    if endpoints_fallidos:
        for n, p in endpoints_fallidos:
            print(f'║  ❌ {n[:30]:<30} p95={p}ms{" "*(15-len(str(p)))}║')
    else:
        print('║  ✅ Todos los endpoints cumplen el SLA de 500ms              ║')
    print(f"""╠══════════════════════════════════════════════════════════════╣
║  CAUSA RAÍZ IDENTIFICADA                                     ║
║  - /api/notas/buscar: full table scan sin índice en BD       ║
║  - /api/reportes: genera informe completo sin paginación     ║
╠══════════════════════════════════════════════════════════════╣
║  VEREDICTO: {'✅ LISTO PARA PRODUCCIÓN' if listo else '❌ NO ESTÁ LISTO PARA PRODUCCIÓN':<46}║
╠══════════════════════════════════════════════════════════════╣
║  RECOMENDACIONES                                             ║
║  1. Agregar índice en BD para /buscar (campo 'nombre')       ║
║  2. Paginar el endpoint de reportes (máx 100 filas)          ║
║  3. Agregar caché Redis para /estudiantes (TTL 60s)          ║
║  4. Re-ejecutar tests post-fix para validar mejoras          ║
╚══════════════════════════════════════════════════════════════╝
""")
else:
    print('⚠️ Ejecuta las celdas anteriores primero.')

---
## 📚 Respuestas a las preguntas de reflexión

### 1. ¿Por qué `/api/notas/buscar` es más lento?
El endpoint `/buscar` ejecuta una consulta sin índice en la BD — un **full table scan**. Para cada petición, la BD revisa TODOS los registros. Con N usuarios concurrentes, cada uno espera que la BD termine su escaneo completo antes de atender la siguiente consulta. La solución es agregar un índice B-tree en el campo de búsqueda (`CREATE INDEX idx_nombre ON notas(nombre)`), lo que reduce la búsqueda de O(n) a O(log n).

### 2. ¿Por qué no basta el promedio?
Si 49 usuarios tardan 120ms y 1 usuario tarda 8.000ms, el promedio dice ≈280ms. El sistema "parece aceptable". Pero ese usuario está esperando 8 segundos — una experiencia terrible. El p99 revela eso. En un sistema con 1.000 usuarios, el p99 representa 10 personas con mala experiencia en cada segundo de operación.

### 3. Cambios para CI/CD
- Reducir duración: 60s en vez de 5min para no bloquear el pipeline.
- Reducir VUs: 30-50 es suficiente para detectar regresiones graves.
- Criterio de fallo automático: si `p95 > 500ms` o `error_rate > 1%`, el test falla el pipeline.
- Ejecutar solo en PRs a `main`, no en cada feature branch.

### 4. Load Test vs Stress Test
- **Load Test**: valida que el sistema cumple el SLA bajo **carga normal esperada**. Responde: ¿funciona para el caso de uso normal?
- **Stress Test**: encuentra el **límite real** del sistema y cómo falla. Responde: ¿hasta dónde aguanta? ¿falla elegantemente? Es la información que el equipo de infraestructura necesita para planear la capacidad.